In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.initializers import Constant

In [20]:
# Keep the demonstration reproducible
tf.keras.utils.set_random_seed(42)

In [21]:
# One input and one target
X = tf.constant([[1.0]], dtype=tf.float32)
y = tf.constant([[0.0]], dtype=tf.float32)

In [22]:
def build_problematic_model():
    model = Sequential([Input(shape=(1,))])

    # Deliberately use large weights in a deep linear network
    for _ in range(20):
        model.add(
            Dense(
                1,
                activation="linear",
                kernel_initializer=Constant(1.3),
                bias_initializer="zeros"))
    return model

In [23]:
def inspect_gradients(model):
    with tf.GradientTape() as tape:
        predictions = model(X, training=True)
        loss = tf.reduce_mean(tf.square(y - predictions))

    gradients = tape.gradient(loss, model.trainable_variables)
    gradient_norm = tf.linalg.global_norm(gradients)

    print(f"Prediction:    {predictions.numpy().ravel()[0]:.6g}")
    print(f"Loss:          {loss.numpy():.6g}")
    print(f"Gradient norm: {gradient_norm.numpy():.6g}")

    return loss.numpy(), gradient_norm.numpy()

In [24]:
# Model A: no gradient clipping
model_without_clipping = build_problematic_model()


print("\n--- Before training: No clipping ---")
loss_before_a, norm_before_a = inspect_gradients(model_without_clipping)

model_without_clipping.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=1e-4), loss="mse")

model_without_clipping.fit(X, y, epochs=1, verbose=0)

print("\n--- After one update: No clipping ---")
inspect_gradients(model_without_clipping)


--- Before training: No clipping ---
Prediction:    190.05
Loss:          36118.8
Gradient norm: 263282

--- After one update: No clipping ---
Prediction:    8.03381e+12
Loss:          6.45422e+25
Gradient norm: inf


(np.float32(6.454215e+25), np.float32(inf))

In [25]:
# Model B: gradient clipping
model_with_clipping = build_problematic_model()

print("\n--- Before training: With clipping ---")
loss_before_b, norm_before_b = inspect_gradients(model_with_clipping)

model_with_clipping.compile(optimizer=tf.keras.optimizers.SGD(
        learning_rate=1e-4,
        global_clipnorm=1.0),
    loss="mse")

model_with_clipping.fit(X, y, epochs=1, verbose=0)

print("\n--- After one update: With clipping ---")
inspect_gradients(model_with_clipping)


--- Before training: With clipping ---
Prediction:    190.05
Loss:          36118.8
Gradient norm: 263282

--- After one update: With clipping ---
Prediction:    189.98
Loss:          36092.5
Gradient norm: 263097


(np.float32(36092.516), np.float32(263097.12))